In [1]:
%cd ../
%ls

/Users/delapazm/Desktop/wellcome_academic_graph_toolkit/venv/lib/python3.9/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/Users/delapazm/Desktop/wellcome_academic_graph_toolkit
ID_Accelerate/   dist/            idr/             wag_toolkit/
Makefile         edges_all.json   nodes_all.json
README.md        environment.yml  pyproject.toml
careers/         geographies/     venv/


In [2]:
from wag_toolkit.locations import Locations
import pandas as pd
import json
from dotenv import load_dotenv
import awswrangler as wr
import os
import shutil
import numpy as np

load_dotenv()

True

### Load Grants

In [3]:
grants_ref =  wr.s3.read_csv('s3://datalabs-data/funding_impact_measures/infectious_disease/enriched_grants.csv', encoding='latin1')
grants_ref = grants_ref[grants_ref['strategic_goal']== '1. Accelerate']
grants_ids = list(set(grants_ref['grant_reference'].to_list()))

### Load Pubs 

In [4]:
pubs_orgs_df = wr.s3.read_parquet('s3://datalabs-data/funding_impact_measures/infectious_disease/enriched_publications_orgs.parquet')
pubs_orgs_df = pubs_orgs_df[['dimensions_publication_id', 'publication_date', 'research_orgs', 'supporting_grant_ids']]
pubs_orgs_df = pubs_orgs_df.explode('supporting_grant_ids')

id_grant_pubs = grants_ref.merge(pubs_orgs_df, how='inner', on='supporting_grant_ids')
id_grant_pubs['dimensions_publication_id'] = id_grant_pubs['dimensions_publication_id'].apply(lambda x: 'pub.' + str(x))
id_grant_pubs['year'] = pd.to_datetime(id_grant_pubs['publication_date'], errors='coerce').dt.year
id_grant_pubs = id_grant_pubs.explode('research_orgs')
id_grant_pubs['grid_id'] = id_grant_pubs['research_orgs'].apply(lambda x: x['id'] if isinstance(x, dict) else x)
id_grant_pubs = id_grant_pubs[['dimensions_publication_id', 'year', 'grid_id']]
unique_grant_pubs = id_grant_pubs.drop_duplicates(subset=['dimensions_publication_id', 'grid_id'])

### Initialise Locations Class

In [5]:
dummy_query = """MATCH (r:Researcher)-[a:AUTHORED]->(p:Publication) RETURN * LIMIT 1"""
loc = Locations(dummy_query)
loc.data = unique_grant_pubs.to_dict(orient='records')

100%|██████████| 1/1 [00:00<00:00,  2.79it/s]


In [6]:
loc._clean_grid_ids()
loc.extract_edges()
loc.extract_locations("country")

loc.convert_edges()
loc.calculate_adjacency_matrices()

100%|██████████| 1/1 [00:00<00:00,  7.60it/s]
/Users/delapazm/Desktop/wellcome_academic_graph_toolkit/wag_toolkit/locations.py:186: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.crosstab(
/Users/delapazm/Desktop/wellcome_academic_graph_toolkit/wag_toolkit/locations.py:186: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.crosstab(
/Users/delapazm/Desktop/wellcome_academic_graph_toolkit/wag_toolkit/locations.py:186: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep cu

2024.0
2023.0
2022.0
2021.0
2020.0
2025.0
2019.0
2018.0
2017.0
2015.0
2016.0


/Users/delapazm/Desktop/wellcome_academic_graph_toolkit/wag_toolkit/locations.py:186: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.crosstab(
/Users/delapazm/Desktop/wellcome_academic_graph_toolkit/wag_toolkit/locations.py:186: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.crosstab(
/Users/delapazm/Desktop/wellcome_academic_graph_toolkit/wag_toolkit/locations.py:186: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.


In [7]:
loc.adjacency_matrices = {np.int64(key): value for key, value in loc.adjacency_matrices.items()}

### Load Countries Data with Income Classification

In [8]:
countries_class = pd.read_excel('geographies/files/CLASS.xlsx', sheet_name='List of economies')
lmic_list = countries_class[countries_class['Income group'].isin(['Low income', 'Lower middle income', 'Upper middle income'])]

with open('geographies/files/country_modifications.json', 'r') as file:
    modifications = json.load(file)["modifications"]

for old_name, new_name in modifications.items():
    lmic_list['Economy'] = lmic_list['Economy'].replace(old_name, new_name)

/var/folders/dy/p3vg7l610wx7yc4y2908psy00000gp/T/ipykernel_44216/296399124.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lmic_list['Economy'] = lmic_list['Economy'].replace(old_name, new_name)


### Filter by LMIC - Optional

In [ ]:
only_lmic = False

if only_lmic:
    for year in loc.adjacency_matrices:
        for country in loc.adjacency_matrices[year]['All'].index:
            if country == 'All' or country == 'total':
                continue
            for country2 in loc.adjacency_matrices[year]['All'][country].index:
                if country in list(lmic_list['Economy']) or country2 in list(lmic_list['Economy']):
                    continue
                else:
                    loc.adjacency_matrices[year]['All'][country][country2]=0

### VisJS

In [10]:
loc.load_visjs_nodes_and_edges(node_scaling=0.0015, edge_scaling=0.05, directed=False, threshold=0, font = {"size": 20, "face": "Helvetica Neue"}, node_count='total')

In [11]:
dirname = 'ID_Accelerate'
loc.to_visjs(vis_name="locations", directed=False, template='geographies/locations.html', dirname=dirname)
loc._to_json("nodes_all.json", loc.vis_nodes)
loc._to_json("edges_all.json", loc.vis_edges)

In [12]:
with(open(f'{dirname}/edges.json','r')) as f:
    edges = json.load(f)

os.rename(f'{dirname}/nodes.json', f'{dirname}/nodes_all.json')
shutil.copyfile('geographies/positions.json', f'{dirname}/positions.json')

def edges_to_dict(edges):
    edge_dict = {}
    for e in edges:
        year = e['year']
        _ = e.pop('year')
        if edge_dict.get(year):
            edge_dict[year].append(e)
        else:
            edge_dict[year] = [e]
    return edge_dict


with(open(f'{dirname}/dict_edges_all.json','w')) as f:
        json.dump(edges_to_dict(edges), f)